In [1]:
import pandas as pd
import torch
import torch.nn as nn
import torch.optim as optim
from sklearn.model_selection import train_test_split
from transformers import AutoTokenizer

In [3]:
# 1. Load only 3,000 rows for lab-speed training
df = pd.read_csv("IMDB Dataset.csv", nrows=3000)

# Convert sentiment into numerical labels
df["sentiment"] = df["sentiment"].map({
    "positive": 1,
    "negative": 0
})

print(df.head())
print(df.shape)

                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...          0
4  Petter Mattei's "Love in the Time of Money" is...          1
(3000, 2)


In [4]:
# 3. Split into training and test sets
X_train, X_test, y_train, y_test = train_test_split(
    df["review"].tolist(),
    df["sentiment"].tolist(),
    test_size=0.2,
    random_state=42,
    stratify=df["sentiment"]
)

In [5]:
# 4. Tokenize review texts using tiktoken cl100k_base encoding
import tiktoken

encoding = tiktoken.get_encoding("cl100k_base")

train_tokens = [encoding.encode(text) for text in X_train]
test_tokens = [encoding.encode(text) for text in X_test]

In [6]:
# 5. Pad or truncate sequences to a fixed length
MAX_LEN = 200

def pad_or_truncate(sequences, max_len):
    result = []

    for seq in sequences:
        seq = seq[:max_len]

        if len(seq) < max_len:
            seq = seq + [0] * (max_len - len(seq))

        result.append(seq)

    return result

X_train_padded = pad_or_truncate(train_tokens, MAX_LEN)
X_test_padded = pad_or_truncate(test_tokens, MAX_LEN)

In [7]:
# 6. Convert token sequences and labels into tensors
X_train_tensor = torch.tensor(X_train_padded, dtype=torch.long)
X_test_tensor = torch.tensor(X_test_padded, dtype=torch.long)

y_train_tensor = torch.tensor(y_train, dtype=torch.long)
y_test_tensor = torch.tensor(y_test, dtype=torch.long)

In [8]:
# 7. Build model: Embedding -> RNN -> Linear output layer
VOCAB_SIZE = encoding.n_vocab
EMBED_DIM = 64
HIDDEN_DIM = 128
OUTPUT_DIM = 2

class RNNClassifier(nn.Module):
    def __init__(self):
        super(RNNClassifier, self).__init__()

        self.embedding = nn.Embedding(
            VOCAB_SIZE,
            EMBED_DIM,
            padding_idx=0
        )

        self.rnn = nn.RNN(
            EMBED_DIM,
            HIDDEN_DIM,
            batch_first=True
        )

        self.fc = nn.Linear(HIDDEN_DIM, OUTPUT_DIM)

    def forward(self, x):
        x = self.embedding(x)
        _, hidden = self.rnn(x)
        output = self.fc(hidden[-1])
        return output

model = RNNClassifier()

In [9]:
# 8. Train the model
criterion = nn.CrossEntropyLoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

EPOCHS = 50
BATCH_SIZE = 32

for epoch in range(EPOCHS):

    model.train()
    total_loss = 0

    for i in range(0, len(X_train_tensor), BATCH_SIZE):

        X_batch = X_train_tensor[i:i + BATCH_SIZE]
        y_batch = y_train_tensor[i:i + BATCH_SIZE]

        optimizer.zero_grad()

        outputs = model(X_batch)

        loss = criterion(outputs, y_batch)

        loss.backward()
        optimizer.step()

        total_loss += loss.item()

    print(
        f"Epoch {epoch + 1}/{EPOCHS}, "
        f"Loss: {total_loss / (len(X_train_tensor) / BATCH_SIZE):.4f}"
    )

Epoch 1/50, Loss: 0.7060
Epoch 2/50, Loss: 0.6762
Epoch 3/50, Loss: 0.6412
Epoch 4/50, Loss: 0.5931
Epoch 5/50, Loss: 0.5541
Epoch 6/50, Loss: 0.5098
Epoch 7/50, Loss: 0.5079
Epoch 8/50, Loss: 0.5922
Epoch 9/50, Loss: 0.6247
Epoch 10/50, Loss: 0.6343
Epoch 11/50, Loss: 0.5719
Epoch 12/50, Loss: 0.5160
Epoch 13/50, Loss: 0.4672
Epoch 14/50, Loss: 0.4261
Epoch 15/50, Loss: 0.3951
Epoch 16/50, Loss: 0.3678
Epoch 17/50, Loss: 0.3463
Epoch 18/50, Loss: 0.3331
Epoch 19/50, Loss: 0.3389
Epoch 20/50, Loss: 0.3177
Epoch 21/50, Loss: 0.3321
Epoch 22/50, Loss: 0.3711
Epoch 23/50, Loss: 0.3098
Epoch 24/50, Loss: 0.3009
Epoch 25/50, Loss: 0.3267
Epoch 26/50, Loss: 0.3351
Epoch 27/50, Loss: 0.3595
Epoch 28/50, Loss: 0.3075
Epoch 29/50, Loss: 0.3116
Epoch 30/50, Loss: 0.3134
Epoch 31/50, Loss: 0.2998
Epoch 32/50, Loss: 0.2914
Epoch 33/50, Loss: 0.2799
Epoch 34/50, Loss: 0.2798
Epoch 35/50, Loss: 0.3061
Epoch 36/50, Loss: 0.3383
Epoch 37/50, Loss: 0.3612
Epoch 38/50, Loss: 0.3204
Epoch 39/50, Loss: 0.

In [10]:
print(df.head())

                                              review  sentiment
0  One of the other reviewers has mentioned that ...          1
1  A wonderful little production. <br /><br />The...          1
2  I thought this was a wonderful way to spend ti...          1
3  Basically there's a family where a little boy ...          0
4  Petter Mattei's "Love in the Time of Money" is...          1


In [17]:
# 9. Evaluate RNN: Accuracy, Precision, Recall and Confusion Matrix

model.eval()

with torch.no_grad():

    # Test accuracy
    test_outputs = model(X_test_tensor)

    test_preds = torch.argmax(
        test_outputs,
        dim=1
    )

    test_accuracy = (
        (test_preds == y_test_tensor)
        .float()
        .mean()
        .item()
    )

    # Train accuracy
    train_outputs = model(X_train_tensor)

    train_preds = torch.argmax(
        train_outputs,
        dim=1
    )

    train_accuracy = (
        (train_preds == y_train_tensor)
        .float()
        .mean()
        .item()
    )

print(f"RNN Train Accuracy: {train_accuracy * 100:.2f}%")
print(f"RNN Test Accuracy: {test_accuracy * 100:.2f}%")

from sklearn.metrics import classification_report, confusion_matrix

print(
    classification_report(
        y_test_tensor.numpy(),
        test_preds.numpy(),
        target_names=["negative", "positive"]
    )
)

print("RNN Confusion Matrix:")
print(
    confusion_matrix(
        y_test_tensor.numpy(),
        test_preds.numpy()
    )
)

RNN Train Accuracy: 83.58%
RNN Test Accuracy: 49.33%
              precision    recall  f1-score   support

    negative       0.49      0.36      0.41       298
    positive       0.50      0.63      0.55       302

    accuracy                           0.49       600
   macro avg       0.49      0.49      0.48       600
weighted avg       0.49      0.49      0.48       600

RNN Confusion Matrix:
[[107 191]
 [113 189]]


In [19]:
# Test on new review sentences

new_reviews = [
    "This movie was absolutely amazing and I loved every minute of it.",
    "The movie was boring and a complete waste of time.",
    "The acting was excellent and the story was very interesting.",
    "I hated this movie because it was too slow and predictable."
]

new_tokens = [
    encoding.encode(text)
    for text in new_reviews
]

new_padded = pad_or_truncate(
    new_tokens,
    MAX_LEN
)

new_tensor = torch.tensor(
    new_padded,
    dtype=torch.long
)

model.eval()

with torch.no_grad():

    outputs = model(new_tensor)

    predictions = torch.argmax(
        outputs,
        dim=1
    )

for review, prediction in zip(new_reviews, predictions):

    sentiment = (
        "positive"
        if prediction.item() == 1
        else "negative"
    )

    print(f"Review: {review}")
    print(f"RNN Prediction: {sentiment}")
    print()

Review: This movie was absolutely amazing and I loved every minute of it.
RNN Prediction: positive

Review: The movie was boring and a complete waste of time.
RNN Prediction: positive

Review: The acting was excellent and the story was very interesting.
RNN Prediction: positive

Review: I hated this movie because it was too slow and predictable.
RNN Prediction: positive

